In [1]:
import duckdb

con = duckdb.connect()
con.sql("SET memory_limit='4GB'")

con.sql("""
SELECT code, brands, categories_tags, countries_tags,
       nutriscore_grade, nutriscore_score, nutriments, product_name
FROM '../data/food.parquet'
LIMIT 2
""").df()

,code,brands,categories_tags,countries_tags,nutriscore_grade,nutriscore_score,nutriments,product_name
0,0000101209159,Bovetti,"[en:breakfasts, en:spreads, en:sweet-spreads, ...",[en:france],e,25,"[{'name': 'sugars', 'value': None, '100g': 32....","[{'lang': 'main', 'text': 'Véritable pâte à ta..."
1,0000105000011,Lagg's,[en:null],[en:united-states],unknown,<NA>,[{'name': 'fruits-vegetables-legumes-estimate-...,"[{'lang': 'main', 'text': 'Chamomile Herbal Te..."


In [3]:
con.sql("SET threads=2")

con.sql("""
COPY (
    SELECT code, product_name, brands, brands_tags, categories_tags,
           countries_tags, nutriscore_grade, nutriscore_score,
           nutriments, no_nutrition_data, nutrition_data_per
    FROM '../data/food.parquet'
) TO '../data/bronze_off.parquet' (FORMAT parquet, COMPRESSION zstd)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [4]:
import os

taille = os.path.getsize('../data/bronze_off.parquet') / 1e9
print(f"Taille du fichier bronze : {taille:.2f} Go")

Taille du fichier bronze : 0.41 Go


In [5]:
con.sql("""
SELECT COUNT(*) AS nb_produits
FROM '../data/bronze_off.parquet'
""").show()

┌─────────────┐
│ nb_produits │
│    int64    │
├─────────────┤
│     4882161 │
└─────────────┘



In [6]:
con.sql("""
COPY (
    SELECT * EXCLUDE (nutriments),
           to_json(nutriments) AS nutriments_json
    FROM '../data/bronze_off.parquet'
) TO '../data/bronze_off_bq.parquet' (FORMAT parquet, COMPRESSION zstd)
""")

import os
print(f"{os.path.getsize('../data/bronze_off_bq.parquet') / 1e9:.2f} Go")

con.sql("SELECT COUNT(*) AS nb_produits FROM '../data/bronze_off_bq.parquet'").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

0.70 Go
┌─────────────┐
│ nb_produits │
│    int64    │
├─────────────┤
│     4882161 │
└─────────────┘



In [7]:
%pip install google-cloud-bigquery pydata-google-auth


     ------------------------------------ 275.4/275.4 KB 940.2 kB/s eta 0:00:00
     -------------------------------------- 224.3/224.3 KB 2.3 MB/s eta 0:00:00
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl (229 kB)
     ---------------------------------------- 81.5/81.5 KB 2.3 MB/s eta 0:00:00
     -------------------------------------- 267.9/267.9 KB 1.5 MB/s eta 0:00:00
     -------------------------------------- 50.8/50.8 KB 862.7 kB/s eta 0:00:00
     -------------------------------------- 456.5/456.5 KB 1.9 MB/s eta 0:00:00
     ---------------------------------------- 60.3/60.3 KB 1.1 MB/s eta 0:00:00
     -------------------------------------- 307.7/307.7 KB 2.1 MB/s eta 0:00:00
     ---------------------------------------- 5.6/5.6 MB 2.5 MB/s eta 0:00:00
     ---------------------------------------- 3.9/3.9 MB 3.6 MB/s eta 0:00:00
     -------------------------------------- 181.3/181.3 KB 2.2 MB/s eta 0:00:00
     -------------------------------------- 84.8/84

You should consider upgrading via the 'C:\Users\ayoub\OneDrive\Bureau\.venv\Scripts\python.exe -m pip install --upgrade pip' command.


In [1]:
import pydata_google_auth
from google.cloud import bigquery

credentials = pydata_google_auth.get_user_credentials(
    ["https://www.googleapis.com/auth/bigquery"],
)
client = bigquery.Client(project="off-nutrition-pipeline", credentials=credentials)

print(list(client.query("SELECT 1 AS test").result()))

C:\Users\ayoub\OneDrive\Bureau\.venv\lib\site-packages\google\api_core\_python_version_support.py:212: FutureWarning: You are using a non-supported Python version (3.10.5). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.11, and then update google.api_core.
  warnings.warn(message, FutureWarning)
C:\Users\ayoub\OneDrive\Bureau\.venv\lib\site-packages\google\api_core\_python_version_support.py:212: FutureWarning: You are using a non-supported Python version (3.10.5). Google will not post any further updates to google.cloud.bigquery supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.11, and then update google.cloud.bigquery.
  warnings.warn(message, FutureWarning)


Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=262006177488-3425ks60hkk80fssi9vpohv88g6q1iqd.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8080%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fbigquery&state=XYhefWHy7J6Usz4AzEZ97g1z65TlF0&code_challenge=HZmkL8BVpnVGZcMtEMC51TO5KMuJsw4KkJ6_PV9yyOw&code_challenge_method=S256&prompt=consent&access_type=offline
[Row((1,), {'test': 0})]


In [2]:
job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.PARQUET,
    write_disposition="WRITE_TRUNCATE",
)

with open("../data/bronze_off_bq.parquet", "rb") as f:
    job = client.load_table_from_file(
        f,
        "off-nutrition-pipeline.bronze.produits",
        job_config=job_config,
        location="EU",
    )

job.result()
print("Chargement terminé")

table = client.get_table("off-nutrition-pipeline.bronze.produits")
print(table.num_rows, "lignes")

Chargement terminé
4882161 lignes
